In [ ]:
import geopandas as gpd
from sqlalchemy import create_engine
 
user = "njteh_select"
password = "\\0#IHiT3,s2m"
host = "wirelesspostgresqlflexible.postgres.database.azure.com"
port = "5432"
database = "wiroidb2"
 
connection_string = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}"
 
engine = create_engine(connection_string)

query = "SELECT * FROM us_road_2025 WHERE state_name='NY';"
 
gdf = gpd.GeoDataFrame.from_postgis(
    query,
    engine,
    geom_col="geom"
)
 
gdf.head()

In [ ]:
gdf.to_parquet("edges_NY.parquet", index=False)

In [ ]:
from src.data.graph_builder import GraphBuilder
g_g = GraphBuilder()
G = g_g.build_graph(roads = gdf)

In [ ]:
g_g = GraphBuilder()
G = g_g.build_graph(roads = gdf)

In [ ]:
# assume `roads_gdf` is huge, and `locations` is your points GeoDataFrame
bbox = locations.total_bounds  # [minx, miny, maxx, maxy]
buffer_m = 1000  # how far out to keep roads
# convert buffer into degrees approx (roughly 1km ~= 0.009 degrees)
buffer_deg = buffer_m / 111000.0

minx, miny, maxx, maxy = bbox
roads_sub = roads_gdf.cx[minx-buffer_deg : maxx+buffer_deg, miny-buffer_deg : maxy+buffer_deg]

In [ ]:
pip install osmnx==1.3.0

In [ ]:
from osmnx.utils_graph import graph_from_gdfs
import pandas as pd
import networkx as nx

def create_graph(road_data):
        
        road_data["start_point"] = road_data.geometry.apply(lambda x: x.coords[0])

        road_data["end_point"] = road_data.geometry.apply(lambda x: x.coords[-1])

        gdf_nodes = pd.DataFrame({"data" : list(set(list(set(road_data["start_point"]))+ list(set(road_data["end_point"]))))})

        gdf_nodes["x"] = gdf_nodes["data"].apply(lambda x : x[0])

        gdf_nodes["y"] = gdf_nodes["data"].apply(lambda x : x[1])

        gdf_nodes["osmid"]= gdf_nodes.index

        dictt= gdf_nodes.set_index("data")["osmid"].to_dict()

        road_data["u"] = road_data["start_point"].map(dictt)

        road_data["v"] = road_data["end_point"].map(dictt)

        road_data["key"] = 0

        gdf_edges = road_data[["u","v","key","geometry"]]

        gdf_edges.set_index(['u', 'v', 'key'], inplace=True)

        gdf_edges["length"] = gdf_edges.geometry.length

        G = graph_from_gdfs(gdf_nodes,gdf_edges)

        G = nx.Graph(G)
        
        return G,gdf_nodes,gdf_edges


In [ ]:
gdf = gdf.explode()

In [ ]:
gdf = gdf.reset_index(drop=True)

In [ ]:
gdf = gdf.rename(columns = {"geom" : "geometry"})
gdf.set_geometry("geometry", inplace=True)

In [ ]:
G, gdf_nodes, gdf_edges = create_graph(road_data = gdf)

In [ ]:
# inspect dtypes
print(gdf_nodes.dtypes)

# convert any pandas Period columns to string
for col, dt in gdf_nodes.dtypes.items():
    if str(dt).startswith("period"):
        gdf_nodes[col] = gdf_nodes[col].astype(str)

In [ ]:
from pandas.api.types import is_period_dtype

# Select only the columns we want to export
# Convert any object/period-like columns to safe types for Parquet

d = pd.DataFrame({
    # "data": gdf_nodes["da114ta"],
    "x": gdf_nodes["x"],
    "y": gdf_nodes["y"],
    "osmid": gdf_nodes["osmid"],
})

print("Export dtypes before conversion:")
print(d.dtypes)

for col in d.columns:
    dt = d[col].dtype
    if is_period_dtype(dt):
        d[col] = d[col].astype(str)
    elif dt == object:
        # If objects contain pandas Period values, convert to strings
        sample = d[col].dropna().head(20)
        if any(isinstance(v, pd.Period) for v in sample):
            d[col] = d[col].astype(str)

print("Export dtypes after conversion:")
print(d.dtypes)

# Try pyarrow first, fall back to fastparquet if needed
try:
    d.to_parquet("nodes_NY.parquet", index=False, engine="pyarrow")
    print("Exported nodes_NY.parquet with pyarrow")
except Exception as e:
    print("pyarrow failed, falling back to fastparquet:", e)
    try:
        d.to_parquet("nodes_NY.parquet", index=False, engine="fastparquet")
        print("Exported nodes_NY.parquet with fastparquet")
    except Exception as e2:
        print("fastparquet also failed:", e2)
        raise

In [ ]:
import pandas as pd

# nodes_df = pd.read_parquet("nodes_NY.parquet", engine="pyarrow")
edges_df = pd.read_parquet("edges_NY.parquet", engine="pyarrow")

In [ ]:
import pandas as pd

nodes_gdf = pd.read_parquet("nodes_NY.parquet", engine="pyarrow")
edges_gdf = pd.read_parquet("edges_NY.parquet", engine="pyarrow")

In [ ]:
edges_gdf

In [ ]:
nodes_gdf

In [ ]:
import geopandas as gpd
import shapely 
edges_gdf['geometry'] = edges_gdf['geometry'].apply(lambda x: shapely.from_wkb(x))

In [ ]:
gpd.GeoDataFrame(edges_gdf).to_file("edges_NY.geojson", driver="GeoJSON")